# Tutorial prático: análise de dados e previsão de preços de casas 🏠

**Pandas • Matplotlib • scikit-learn • conjunto de dados House Prices (Kaggle)**

Neste notebook, vamos sair da leitura de um arquivo CSV até a construção e avaliação de um modelo de **regressão linear** para estimar o preço de venda de imóveis.

**Ao terminar, você saberá:**
1. Carregar e explorar tabelas com **Pandas**.
2. Consultar colunas, filtrar linhas, calcular estatísticas e tratar valores ausentes.
3. Visualizar a relação entre variáveis com **Matplotlib**.
4. Separar dados de **treino, validação e teste**.
5. Criar variáveis, transformar categorias em números e preparar os dados.
6. Treinar uma **LinearRegression** com scikit-learn e avaliar o **erro absoluto médio (MAE)**.

> **Como estudar:** execute as células de código **de cima para baixo**. Leia os textos antes de cada exemplo, examine as saídas e tente responder às perguntas propostas. Os números das métricas dependem dos dados e da divisão escolhida; usamos `random_state=42` para tornar a divisão reproduzível.

## Roteiro da aula

1. **Preparação e carregamento** — bibliotecas, CSV e DataFrame.
2. **Exploração com Pandas** — consultas, resumos, filtros e valores faltantes.
3. **Visualização com Matplotlib** — dispersão e distribuição de preços.
4. **Divisão dos dados** — treino × validação × teste.
5. **Engenharia de atributos e tratamento** — idade do imóvel, imputação e one-hot encoding.
6. **Regressão linear e avaliação** — treinamento, previsões, baseline e MAE.
7. **Desafios e referências** — sugestões para continuar praticando.

**Pré-requisitos:** noções básicas de variáveis e listas em Python. O tutorial funciona no **Google Colab** ou em Jupyter com `pandas`, `matplotlib` e `scikit-learn` instalados.

Caso seja necessário instalar as bibliotecas, execute em uma célula separada do seu ambiente:

```python
%pip install pandas matplotlib scikit-learn
```

---
## 1. Preparação do ambiente

Vamos importar as bibliotecas que serão usadas ao longo da aula:

- **Pandas (`pd`)**: lê e manipula dados organizados em linhas e colunas.
- **Matplotlib (`plt`)**: constrói gráficos.
- **scikit-learn**: oferece funções para separar dados, treinar modelos e calcular métricas.

O comando `import ... as ...` cria um *apelido* curto para a biblioteca; por exemplo, escrever `pd.read_csv(...)` em vez de `pandas.read_csv(...)`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

print("Bibliotecas importadas com sucesso!")

### 1.1 O conjunto de dados

Usaremos os dados da competição **[House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)**, do Kaggle. Cada linha descreve uma casa; a coluna **`SalePrice`** contém seu preço de venda (em dólares dos EUA).

Neste material, o CSV é lido de um **espelho público no GitHub**, assim como no notebook original. É necessário ter acesso à internet na primeira leitura. Como alternativa, baixe o arquivo `train.csv` na página do Kaggle e coloque-o na mesma pasta deste notebook (ou envie-o ao Colab).

> **Atenção:** o arquivo `train.csv` contém os preços reais, apropriados para estudar aprendizado supervisionado. O `test.csv` disponibilizado na competição é outro arquivo e não deve ser confundido com o **conjunto de teste** que criaremos abaixo.

In [ ]:
from pathlib import Path

url = "https://raw.githubusercontent.com/Rswcf/houseprices_kaggle/master/train.csv"

# Procura primeiro um arquivo local, para funcionar também sem conexão.
caminhos_locais = [Path("train.csv"), Path("/content/train.csv")]
arquivo_local = next((p for p in caminhos_locais if p.is_file()), None)

try:
    df = pd.read_csv(arquivo_local if arquivo_local is not None else url)
except Exception as erro:
    raise RuntimeError(
        "Não foi possível carregar os dados. Baixe 'train.csv' no Kaggle "
        "e deixe-o ao lado do notebook (ou em /content no Colab)."
    ) from erro

print(f"Dados carregados: {df.shape[0]} linhas e {df.shape[1]} colunas")

### 1.2 O que é um DataFrame?

Um **DataFrame** é uma tabela do Pandas: cada **linha** representa uma observação (uma casa) e cada **coluna** representa uma característica (por exemplo, área, bairro ou preço).

- `df.shape`: retorna uma tupla com **(linhas, colunas)**.
- `df.head()`: mostra as **5 primeiras linhas**, por padrão.
- `df.head(1)`: mostra apenas a **primeira linha**.
- `df.columns`: informa os nomes das colunas.

Exibir apenas parte da tabela costuma ser mais útil do que imprimir todas as linhas de uma vez.

In [ ]:
print("Dimensão:", df.shape)
display(df.head())
display(df.head(1))

In [ ]:
# Uma lista com os nomes das colunas disponíveis.
print(df.columns.tolist())

---
## 2. Explorando e filtrando dados com Pandas

Uma análise inicial serve para entender **que informações existem** e **como estão representadas**. Vamos começar selecionando quatro colunas do conjunto original:

| Coluna | Significado |
| --- | --- |
| `SalePrice` | Preço final de venda (US$). |
| `GrLivArea` | Área habitável acima do nível do solo (pés²). |
| `Neighborhood` | Bairro da propriedade (categoria). |
| `GarageCars` | Capacidade da garagem, em número de carros. |

Para selecionar várias colunas, usamos **dois pares de colchetes**: `df[["coluna_1", "coluna_2"]]`.

In [ ]:
colunas_exemplo = ["SalePrice", "GrLivArea", "Neighborhood", "GarageCars"]
display(df[colunas_exemplo].head())

### 2.1 Resumos estatísticos

`describe()` calcula estatísticas **das colunas numéricas** por padrão:

- `count`: quantos valores não estão ausentes;
- `mean`: média;
- `std`: desvio padrão (dispersão dos dados);
- `min` e `max`: menores e maiores valores;
- `25%`, `50%`, `75%`: quartis. O **50%** corresponde à **mediana**.

Como `Neighborhood` é uma categoria (texto), ela não aparece na tabela de estatísticas numéricas. Podemos analisá-la separadamente com `value_counts()`.

In [ ]:
display(df[colunas_exemplo].describe())

print("Bairros mais frequentes:")
display(df["Neighborhood"].value_counts().head(10))

### 2.2 Cálculos e filtros

Uma coluna sozinha, como `df["SalePrice"]`, é uma **Series** do Pandas. Podemos chamar métodos nela, por exemplo `.sum()`.

Para filtrar linhas usamos `df.loc[condição]`. A condição `df["SalePrice"] < 50000` devolve uma sequência de valores verdadeiros/falsos; `.loc` mantém apenas as linhas que satisfazem a condição.

> A soma dos preços é apenas um exemplo de operação numérica: **não** representa receita de uma empresa, pois a tabela reúne vendas de diferentes imóveis.

In [ ]:
soma_precos = df["SalePrice"].sum()
casas_abaixo_50mil = df.loc[df["SalePrice"] < 50_000]

print(f"Soma dos preços de venda: US$ {soma_precos:,.2f}")
print("Casas vendidas por menos de US$ 50.000:", len(casas_abaixo_50mil))
display(casas_abaixo_50mil[["SalePrice", "GrLivArea", "Neighborhood"]].head())

### 2.3 Valores ausentes (`NaN`)

Dados reais frequentemente têm informações não preenchidas. `isna()` identifica valores faltantes e `sum()` contabiliza quantos existem em cada coluna. Isso importa porque muitos algoritmos de aprendizado de máquina não aceitam `NaN` diretamente.

Duas estratégias comuns são:

- **Mediana** para colunas numéricas, especialmente quando há valores muito altos ou baixos.
- **Moda** (valor mais comum) ou um rótulo como `"Desconhecido"` para colunas categóricas.

**Detalhe importante:** `fillna()` devolve uma versão preenchida; ele **não altera necessariamente a coluna original**. Por isso atribuímos o resultado explicitamente a uma variável. Mais adiante, calcularemos os valores de preenchimento **somente com dados de treino**, para evitar vazamento de informação (*data leakage*).

In [ ]:
faltantes = df.isna().sum().sort_values(ascending=False)
print("Colunas com mais valores ausentes:")
display(faltantes.head(12))

In [ ]:
# Demonstração de preenchimento pela moda em uma cópia, sem alterar df.
bairro_mais_comum = df["Neighborhood"].mode()[0]
bairro_preenchido = df["Neighborhood"].fillna(bairro_mais_comum)

print("Bairro mais frequente:", bairro_mais_comum)
print("Nulos antes:", df["Neighborhood"].isna().sum())
print("Nulos na série preenchida:", bairro_preenchido.isna().sum())

**Teste seu entendimento:** qual a diferença entre `df["SalePrice"]` e `df[["SalePrice"]]`? A primeira expressão retorna uma **Series** (uma dimensão); a segunda, um **DataFrame** (duas dimensões).

---
## 3. Visualizando os dados com Matplotlib

Gráficos tornam padrões mais fáceis de enxergar. Um **gráfico de dispersão** (*scatter plot*) mostra cada casa como um ponto:

- Eixo horizontal (x): área habitável acima do solo, `GrLivArea`.
- Eixo vertical (y): preço de venda, `SalePrice`.

Se pontos com maior área tendem a ter preços maiores, isso sugere uma **associação positiva**. Associação não significa, por si só, causalidade nem garante previsões precisas.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["GrLivArea"], df["SalePrice"], alpha=0.45)
plt.title("Área habitável × preço de venda")
plt.xlabel("Área habitável acima do solo (pés²)")
plt.ylabel("Preço de venda (US$)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 3.1 Como interpretar a dispersão?

Observe se os pontos formam uma tendência aproximadamente crescente e se há **valores extremos** (*outliers*), como casas muito grandes com preços incomuns. Um modelo simples pode ser influenciado por esses pontos.

Podemos também usar um **histograma** para visualizar a distribuição dos preços: cada barra representa uma faixa de valores e sua frequência. Diferentemente do gráfico de dispersão, aqui estudamos **uma variável de cada vez**.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["SalePrice"], bins=30, edgecolor="white")
plt.title("Distribuição dos preços de venda")
plt.xlabel("Preço de venda (US$)")
plt.ylabel("Quantidade de casas")
plt.tight_layout()
plt.show()

---
## 4. Separando os dados para aprendizado de máquina

Se avaliarmos um modelo nos **mesmos dados usados para treiná-lo**, podemos ter uma impressão exageradamente otimista do seu desempenho. Por isso, separamos três conjuntos:

| Conjunto | Objetivo | Proporção final neste tutorial |
| --- | --- | ---: |
| **Treino (`train`)** | Ajustar os parâmetros do modelo. | 64% |
| **Validação (`validation`)** | Comparar abordagens e decidir ajustes. | 16% |
| **Teste (`test`)** | Fazer uma avaliação final, uma vez concluídas as decisões. | 20% |

Primeiro vamos demonstrar uma separação simples de **80% / 20%** com `train_test_split()`; em seguida faremos duas divisões para obter os três grupos.

`random_state=42` fixa a aleatoriedade e torna os resultados reproduzíveis. O `test_size=0.2` significa que 20% das linhas são reservadas para o segundo conjunto retornado.

In [ ]:
treino_exemplo, teste_exemplo = train_test_split(
    df, test_size=0.20, random_state=42
)
print("Exemplo — 80% treino:", len(treino_exemplo))
print("Exemplo — 20% teste:", len(teste_exemplo))

### 4.1 Divisão em três conjuntos (64% / 16% / 20%)

A estratégia do notebook original faz duas divisões:

1. **80% desenvolvimento** + **20% teste final**.
2. Dentro dos 80% de desenvolvimento: **80% treino** + **20% validação**.

Portanto, o treino representa `0,8 × 0,8 = 0,64` (64%) de todas as linhas, e a validação representa `0,8 × 0,2 = 0,16` (16%).

Fazemos `.copy()` em cada parte para trabalhar com os conjuntos sem modificações acidentais no DataFrame original.

In [ ]:
dev, test = train_test_split(df, test_size=0.20, random_state=42)
train, validation = train_test_split(dev, test_size=0.20, random_state=42)

train = train.copy()
validation = validation.copy()
test = test.copy()

print(f"Treino:     {len(train):4} casas ({len(train)/len(df):.1%})")
print(f"Validação:  {len(validation):4} casas ({len(validation)/len(df):.1%})")
print(f"Teste:      {len(test):4} casas ({len(test)/len(df):.1%})")

assert len(train) + len(validation) + len(test) == len(df)
assert set(train.index).isdisjoint(validation.index)
assert set(train.index).isdisjoint(test.index)
assert set(validation.index).isdisjoint(test.index)

> **Por que não usar o teste para decidir quais variáveis ou parâmetros usar?** Porque isso faria o modelo ser ajustado indiretamente aos dados de teste, comprometendo a avaliação final. Na prática, a análise exploratória que influencia decisões do modelo deve ser feita no treino/desenvolvimento; aqui examinamos o CSV completo antes da separação apenas para aprender os comandos de Pandas.

---
## 5. Preparando as variáveis do modelo

O algoritmo espera duas partes:

- **`X` (atributos ou *features*)**: informações conhecidas sobre a casa e usadas para estimar o preço.
- **`y` (alvo ou *target*)**: o preço real, `SalePrice`, que queremos prever.

### 5.1 Engenharia de atributos: idade da casa

Criar uma variável a partir de outras chama-se **engenharia de atributos** (*feature engineering*). Vamos estimar a idade do imóvel no momento da venda:

\[
\text{HouseAge} = \text{YrSold} - \text{YearBuilt}
\]

Por exemplo, um imóvel construído em 1990 e vendido em 2009 teria aproximadamente **19 anos** na data da venda. A transformação é aplicada do mesmo modo ao treino, à validação e ao teste.

In [ ]:
for conjunto in (train, validation, test):
    conjunto["HouseAge"] = conjunto["YrSold"] - conjunto["YearBuilt"]

display(train[["YrSold", "YearBuilt", "HouseAge"]].head())

### 5.2 Escolha das características

O notebook original usa cinco variáveis numéricas e duas categóricas:

| Tipo | Variáveis | Interpretação |
| --- | --- | --- |
| Numéricas | `GrLivArea`, `OverallQual`, `GarageCars`, `LotFrontage`, `HouseAge` | Área, qualidade geral, capacidade da garagem, frente do lote e idade. |
| Categóricas | `Neighborhood`, `MSSubClass` | Bairro e tipo/classe da construção. |

Apesar de `MSSubClass` aparecer como um número no CSV, ele é um **código de categoria**; não faz sentido supor que a distância entre seus códigos represente uma diferença quantitativa. Nós o trataremos como texto antes da codificação.

**Importante:** `SalePrice` **não pode** aparecer em `X`, pois isso entregaria ao modelo justamente a resposta que queremos prever.

In [ ]:
numeric_columns = [
    "GrLivArea", "OverallQual", "GarageCars", "LotFrontage", "HouseAge"
]
categorical_columns = ["Neighborhood", "MSSubClass"]
feature_columns = numeric_columns + categorical_columns

# X = colunas explicativas; y = preço real que desejamos prever.
X_train = train[feature_columns].copy()
y_train = train["SalePrice"].copy()

X_validation = validation[feature_columns].copy()
y_validation = validation["SalePrice"].copy()

X_test = test[feature_columns].copy()
y_test = test["SalePrice"].copy()

print("Colunas de entrada:", feature_columns)
display(X_train.head(3))

### 5.3 Imputação: preenchendo os valores numéricos ausentes

O método `median()` calcula a mediana de cada coluna numérica. Usaremos **apenas as medianas do treino** para preencher lacunas nos três conjuntos. Fazer isso evita **vazamento de informação**: as estatísticas da validação e do teste não devem influenciar a etapa de preparação aprendida pelo modelo.

Exemplo: se a mediana de `LotFrontage` no treino for 70, qualquer valor ausente nessa coluna será preenchido por 70 no treino, na validação e no teste.

In [ ]:
print("Valores ausentes nas variáveis numéricas do treino:")
display(X_train[numeric_columns].isna().sum())

median_values = X_train[numeric_columns].median()
print("Medianas calculadas somente no treino:")
display(median_values)

for conjunto_X in (X_train, X_validation, X_test):
    conjunto_X[numeric_columns] = conjunto_X[numeric_columns].fillna(median_values)

### 5.4 Preparando as variáveis categóricas

Algoritmos como a regressão linear do scikit-learn precisam receber valores **numéricos**.

Antes da conversão, vamos:

1. Converter cada categoria em texto (`astype("string")`). Isso é especialmente importante para `MSSubClass`, que é um código representado por número no arquivo original.
2. Usar `fillna("Desconhecido")` para categorias ausentes, preservando uma indicação explícita de que o valor não foi informado.

Em vez de substituir pelo bairro mais comum, essa escolha mantém os ausentes como uma categoria identificável. As duas opções são possíveis; a escolha depende do problema.

In [ ]:
for conjunto_X in (X_train, X_validation, X_test):
    for coluna in categorical_columns:
        conjunto_X[coluna] = (
            conjunto_X[coluna].astype("string").fillna("Desconhecido")
        )

print("Valores de exemplo (antes do one-hot):")
display(X_train[categorical_columns].head())

### 5.5 One-hot encoding: transformando categorias em colunas 0/1

Um bairro não pode ser inserido diretamente como texto em `LinearRegression`. O **one-hot encoding** cria uma coluna indicadora para cada categoria vista no treino.

Exemplo conceitual:

| Neighborhood | Neighborhood_A | Neighborhood_B |
| --- | ---: | ---: |
| A | 1 | 0 |
| B | 0 | 1 |

Com `pd.get_dummies(..., dtype=int)`, os indicadores são representados como **0 e 1**. Já `.reindex(columns=X_train.columns, fill_value=0)` garante que validação e teste tenham **as mesmas colunas e na mesma ordem que o treino**. Uma categoria inédita na validação/teste fica com zeros nos indicadores correspondentes; não adicionamos colunas novas ao modelo.

> Esta é uma abordagem didática e manual. Em aplicações maiores, `Pipeline`, `ColumnTransformer` e `OneHotEncoder(handle_unknown="ignore")` permitem automatizar as etapas e reduzir erros.

In [ ]:
X_train = pd.get_dummies(
    X_train, columns=categorical_columns, dtype=int
)

X_validation = pd.get_dummies(
    X_validation, columns=categorical_columns, dtype=int
).reindex(columns=X_train.columns, fill_value=0)

X_test = pd.get_dummies(
    X_test, columns=categorical_columns, dtype=int
).reindex(columns=X_train.columns, fill_value=0)

print("Atributos após one-hot encoding:")
print("Treino:", X_train.shape)
print("Validação:", X_validation.shape)
print("Teste:", X_test.shape)
display(X_train.iloc[:3, :12])

### 5.6 Checagem antes de treinar

É boa prática confirmar que:

- `X_train`, `X_validation` e `X_test` possuem **as mesmas colunas**;
- **não restaram valores ausentes**;
- o número de linhas de `X` coincide com o número de respostas em `y`.

As instruções `assert` interrompem a execução caso alguma dessas condições não seja verdadeira, ajudando a descobrir problemas cedo.

In [ ]:
assert X_train.columns.equals(X_validation.columns)
assert X_train.columns.equals(X_test.columns)
assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()
assert not X_test.isna().any().any()
assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

print("Preparação conferida: mesmos atributos e nenhum valor ausente!")

---
## 6. Treinando o modelo de regressão linear

A **regressão linear** tenta aproximar o preço de venda por uma combinação dos atributos:

\[
\widehat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_p x_p
\]

- `ŷ` é o **preço previsto**;
- `x₁, x₂, ..., xₚ` são as características do imóvel;
- `b₀` é o intercepto e `b₁, ..., bₚ` são coeficientes aprendidos no treino.

O fluxo básico do scikit-learn segue três verbos: **criar** (`LinearRegression()`), **ajustar** (`fit(X_train, y_train)`) e **prever** (`predict(X_validation)`).

Esse modelo é um **ponto de partida**, não necessariamente a melhor técnica para preços de imóveis: relações complexas, outliers e categorias podem limitar seu desempenho.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

print("Modelo treinado com", X_train.shape[0], "casas e", X_train.shape[1], "atributos.")

### 6.1 Prevendo preços no conjunto de validação

`predict()` devolve uma previsão para cada linha de `X_validation`. Compararemos essas previsões aos valores reais de `y_validation`, que **não foram usados para ajustar os coeficientes**.

In [ ]:
linear_predictions = linear_model.predict(X_validation)

comparacao = pd.DataFrame({
    "Preço real (US$)": y_validation.to_numpy(),
    "Preço previsto (US$)": linear_predictions,
})
comparacao["Erro absoluto (US$)"] = (
    comparacao["Preço real (US$)"] - comparacao["Preço previsto (US$)"]
).abs()

display(comparacao.head(10).round(2))

### 6.2 Avaliando o modelo com MAE

O **erro absoluto médio**, ou **MAE (Mean Absolute Error)**, mede, em média, quanto o valor previsto difere do valor real, **sem considerar o sinal do erro**:

\[
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}\left|y_i - \widehat{y}_iight|
\]

**Interpretação:** um MAE de US$ 20.000 significa que a distância absoluta média entre as previsões e os preços reais é de aproximadamente US$ 20.000. **Quanto menor, melhor**.

Para saber se o modelo está ajudando, comparamos com um **baseline** (referência simples), que prevê o mesmo preço para todas as casas:

- **Baseline da média**: prevê sempre a média dos preços do treino (usado no notebook original).
- **Baseline da mediana**: prevê sempre a mediana dos preços do treino; é uma referência especialmente pertinente ao MAE, pois a mediana minimiza a soma dos erros absolutos em uma amostra.

Os dois usam **somente** a distribuição de preços do treino.

In [ ]:
previsao_media = [y_train.mean()] * len(y_validation)
previsao_mediana = [y_train.median()] * len(y_validation)

baseline_media_mae = mean_absolute_error(y_validation, previsao_media)
baseline_mediana_mae = mean_absolute_error(y_validation, previsao_mediana)
linear_mae = mean_absolute_error(y_validation, linear_predictions)

resultados = pd.DataFrame({
    "Abordagem": ["Baseline: média", "Baseline: mediana", "Regressão linear"],
    "MAE na validação (US$)": [
        baseline_media_mae, baseline_mediana_mae, linear_mae
    ],
}).sort_values("MAE na validação (US$)")

display(resultados.round(2))

### 6.3 Interpretando o resultado

Compare o MAE da regressão com os dois baselines:

- Se o MAE da regressão for **menor**, ela é mais precisa, em média, do que simplesmente repetir um preço constante na validação.
- Se for **maior**, precisamos investigar problemas, como atributos pouco informativos, valores extremos ou inadequação da hipótese de relação linear.

Um gráfico de **preço real × preço previsto** ajuda a identificar padrões. A linha pontilhada representa a situação ideal, em que a previsão coincide com o preço real; quanto mais próximo o ponto estiver dela, menor será o erro daquele imóvel.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_validation, linear_predictions, alpha=0.45)
limite_inferior = min(y_validation.min(), linear_predictions.min())
limite_superior = max(y_validation.max(), linear_predictions.max())
plt.plot(
    [limite_inferior, limite_superior],
    [limite_inferior, limite_superior],
    "--", label="Previsão ideal"
)
plt.xlabel("Preço real (US$)")
plt.ylabel("Preço previsto (US$)")
plt.title("Validação: valores reais × valores previstos")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 6.4 Avaliação final no conjunto de teste

Agora que o procedimento está definido, podemos avaliar o modelo no **teste**, que ficou reservado desde a divisão inicial. O teste oferece uma estimativa de desempenho em casas **não utilizadas no treinamento nem na comparação dos baselines de validação**.

> Em um projeto real, use os resultados do teste para **relatar** o desempenho final; evite ficar alterando o modelo com base neles. Se for necessário continuar experimentando, trabalhe novamente com treino/validação ou use validação cruzada.

In [ ]:
test_predictions = linear_model.predict(X_test)
mae_teste = mean_absolute_error(y_test, test_predictions)
mae_base_teste = mean_absolute_error(
    y_test, [y_train.median()] * len(y_test)
)

print(f"MAE da regressão na validação: US$ {linear_mae:,.2f}")
print(f"MAE da regressão no teste:     US$ {mae_teste:,.2f}")
print(f"MAE do baseline (mediana) no teste: US$ {mae_base_teste:,.2f}")

---
## 7. Hora de praticar: desafios

Tente resolver **sem olhar a solução pronta**. Você pode escrever seus comandos em novas células abaixo dos exemplos. Os três primeiros exercícios usam apenas comandos já apresentados:

1. **Filtros com Pandas:** quantas casas foram vendidas por mais de US$ 300.000? Mostre as 5 primeiras com `SalePrice`, `GrLivArea` e `Neighborhood`.
2. **Estatística descritiva:** encontre a mediana de `GrLivArea` e a média de `SalePrice`. Qual medida é mais sensível a valores muito altos?
3. **Visualização:** faça um gráfico de dispersão relacionando `OverallQual` (eixo x) e `SalePrice` (eixo y). A tendência é crescente?
4. **Desafio extra de modelagem:** treine uma segunda regressão usando **apenas `GrLivArea`** e compare seu MAE na validação com o modelo de vários atributos. Use o mesmo `train` e `validation` já separados.

**Perguntas para refletir:** por que não calculamos a mediana de preenchimento usando validação e teste? Por que o baseline ajuda a interpretar o MAE? Por que não devemos medir a qualidade do modelo apenas nos dados de treino?

In [ ]:
# Espaço para os exercícios 1 a 3 (descomente e complete, se desejar).
# casas_caras = df.loc[df["SalePrice"] > 300_000]
# print("Quantidade:", ...)
# display(...)
# print("Mediana de área:", ...)
# print("Média de preços:", ...)
# plt.scatter(...)
# plt.show()

In [ ]:
# Ponto de partida opcional para o desafio 4.
# Crie a segunda regressão SEM substituir a variável linear_model.
# modelo_area = LinearRegression()
# modelo_area.fit(train[["GrLivArea"]], y_train)
# previsoes_area = modelo_area.predict(validation[["GrLivArea"]])
# print("MAE só com área:", mean_absolute_error(y_validation, previsoes_area))
# print("MAE com vários atributos:", linear_mae)

---
## 8. Revisão e próximos passos

**Fluxo que construímos:**

`CSV → DataFrame → exploração → gráfico → treino/validação/teste → engenharia de atributos → tratamento de ausentes → one-hot encoding → regressão linear → MAE`

**Glossário rápido:**

| Termo | Em outras palavras |
| --- | --- |
| DataFrame | Tabela de linhas e colunas do Pandas. |
| Feature / atributo (`X`) | Informação de entrada utilizada para prever. |
| Target / alvo (`y`) | Resposta conhecida: neste caso, `SalePrice`. |
| Imputação | Preenchimento de valores ausentes. |
| One-hot encoding | Transformação de categorias em colunas indicadoras 0/1. |
| `fit()` | Aprende os coeficientes a partir do treino. |
| `predict()` | Produz previsões para novos dados. |
| MAE | Erro absoluto médio; menor é melhor. |
| Data leakage | Uso indevido de informação da validação/teste no treinamento ou na preparação dos dados. |

**Limitações e possíveis melhorias:** uma regressão linear simples não captura todas as relações entre características e preço; podemos investigar atributos adicionais, regularização, validação cruzada, modelos não lineares e *pipelines*. O desempenho deste notebook é apenas um exercício com um conjunto de dados histórico, não uma avaliação imobiliária atual.

### Referências para continuar estudando

- [Kaggle — House Prices: dados e descrição das variáveis](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)
- [Kaggle Learn — cursos práticos](https://www.kaggle.com/learn)
- [Pandas — documentação](https://pandas.pydata.org/docs/)
- [Matplotlib — galeria de exemplos](https://matplotlib.org/stable/gallery/lines_bars_and_markers/simple_plot.html)
- [scikit-learn — primeiros passos](https://scikit-learn.org/stable/getting_started.html)
- [W3Schools — Python](https://www.w3schools.com/python/)

**Fim do tutorial.** Se quiser revisar, reexecute o notebook do início e tente explicar, com suas palavras, a função de cada etapa.